# 2. Limpieza y transformación de datos

En este cuaderno realizo la limpieza de los datos a partir de lo detectado en la exploración inicial:
1. Estandarizar los nombres de columnas a minúsculas y sin puntos.
2. Corregir columnas numéricas que tienen comas decimales (`euribor3m`, `cons.price.idx`, etc.) para convertirlas a `float`.
3. Limpiar variables de texto (mayúsculas/minúsculas y espacios en blanco).
4. Parsear la columna de fecha en español a formato `datetime` y extraer el mes y el año de contacto.
5. Imputar valores nulos con criterio (edad según profesión, euribor según tasa de empleo, y categorías desconocidas).
6. Crear variables adicionales para el análisis (`total_children`, `previously_contacted`, `age_group`).
7. Guardar el dataset limpio en `data/processed/bank_cleaned.csv`.


In [1]:
import os
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.3f' % x)


### 2.1. Carga y merge de los datos originales
Cargamos los datos y unificamos como hicimos en el paso anterior.


In [2]:
df_campaign = pd.read_csv('../data/raw/bank-additional.csv')

excel_path = '../data/raw/customer-details.xlsx'
xl = pd.ExcelFile(excel_path)
df_customers = pd.concat([pd.read_excel(xl, sheet_name=s) for s in xl.sheet_names], ignore_index=True)

df = pd.merge(df_campaign, df_customers, left_on='id_', right_on='ID', how='inner')
cols_to_drop = [c for c in df.columns if 'Unnamed' in c] + ['ID']
df = df.drop(columns=cols_to_drop)

print(f"Dimensiones iniciales: {df.shape}")


Dimensiones iniciales: (43000, 28)


### 2.2. Normalización de nombres de columnas
Cambiamos los puntos por guiones bajos y convertimos todo a minúsculas para trabajar más cómodos.


In [3]:
df.columns = df.columns.str.lower().str.replace('.', '_')
print(df.columns.tolist())


['age', 'job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'emp_var_rate', 'cons_price_idx', 'cons_conf_idx', 'euribor3m', 'nr_employed', 'y', 'date', 'latitude', 'longitude', 'id_', 'income', 'kidhome', 'teenhome', 'dt_customer', 'numwebvisitsmonth']


### 2.3. Conversión de números con comas decimales
Las columnas macroeconómicas venían con formato europeo (comas en lugar de puntos) y por eso Pandas las tomó como texto. Las convertimos a `float`.


In [4]:
comma_columns = ['cons_price_idx', 'cons_conf_idx', 'euribor3m', 'nr_employed']

for col in comma_columns:
    df[col] = df[col].astype(str).str.replace(',', '.').replace('nan', np.nan).astype(float)

df[comma_columns].dtypes


cons_price_idx    float64
cons_conf_idx     float64
euribor3m         float64
nr_employed       float64
dtype: object

### 2.4. Limpieza de columnas categóricas
Pasamos a minúsculas y quitamos posibles espacios en blanco sobrantes.


In [5]:
text_columns = ['job', 'marital', 'education', 'contact', 'poutcome', 'y']

for col in text_columns:
    df[col] = df[col].astype(str).str.strip().str.lower().replace('nan', np.nan)

print("Valores únicos en marital:", df['marital'].unique())
print("Valores únicos en job:", df['job'].unique()[:5])


Valores únicos en marital: ['married' 'single' 'divorced' nan]
Valores únicos en job: ['housemaid' 'services' 'admin.' 'blue-collar' 'technician']


### 2.5. Tratamiento de fechas
Las fechas vienen en texto con formato `día-mes-año` en español (ejemplo: `2-agosto-2019`).
Mapeo los meses y las convierto a tipo fecha, sacando además el mes y año de contacto.


In [6]:
meses_es = {
    'enero': 1, 'febrero': 2, 'marzo': 3, 'abril': 4, 'mayo': 5, 'junio': 6,
    'julio': 7, 'agosto': 8, 'septiembre': 9, 'octubre': 10, 'noviembre': 11, 'diciembre': 12
}

def parse_spanish_date(val):
    if pd.isna(val) or val == 'nan':
        return pd.NaT
    parts = str(val).split('-')
    if len(parts) == 3:
        day, month_name, year = parts
        month = meses_es.get(month_name.lower())
        if month:
            return pd.Timestamp(year=int(year), month=month, day=int(day))
    return pd.NaT

df['date'] = df['date'].apply(parse_spanish_date)
df['contact_month'] = df['date'].dt.month
df['contact_year'] = df['date'].dt.year

df[['date', 'contact_month', 'contact_year']].dropna().head(3)


,date,contact_month,contact_year
0,2019-08-02,8.000,2019.000
1,2016-09-14,9.000,2016.000
2,2019-02-15,2.000,2019.000


### 2.6. Imputación de nulos

#### A. Edad (`age`)
En vez de rellenar con la media general de 40 años a todo el mundo (lo que distorsionaría a jóvenes o jubilados), imputo la mediana de edad según la profesión (`job`).


In [7]:
job_age_median = df.groupby('job')['age'].transform('median')
global_age_median = df['age'].median()

df['age'] = df['age'].fillna(job_age_median).fillna(global_age_median).astype(int)
print("Nulos en age tras imputar:", df['age'].isna().sum())


Nulos en age tras imputar: 0


#### B. Categóricas (`job`, `marital`, `education`)
Para los casos donde no se registró el dato, asigno la categoría `'unknown'`.


In [8]:
for col in ['job', 'marital', 'education']:
    df[col] = df[col].fillna('unknown')

print("Nulos en categóricas:", df[['job', 'marital', 'education']].isna().sum().to_dict())


Nulos en categóricas: {'job': 0, 'marital': 0, 'education': 0}


#### C. Productos financieros (`default`, `housing`, `loan`)
Relleno los nulos con `-1` para marcar el estado como no informado / desconocido, manteniendo el `0` (no) y `1` (sí).


In [9]:
for col in ['default', 'housing', 'loan']:
    df[col] = df[col].fillna(-1).astype(int)

df['housing'].value_counts()


housing
 1    22498
 0    19476
-1     1026
Name: count, dtype: int64

#### D. Variables macroeconómicas (`euribor3m`, `cons_price_idx`)
El Euribor tiene una correlación muy alta (0.97) con la tasa de variación del empleo (`emp_var_rate`). Imputo los nulos usando la mediana según ese nivel de empleo. Para el IPC imputo la mediana.


In [10]:
euribor_by_emp = df.groupby('emp_var_rate')['euribor3m'].transform('median')
df['euribor3m'] = df['euribor3m'].fillna(euribor_by_emp).fillna(df['euribor3m'].median())
df['cons_price_idx'] = df['cons_price_idx'].fillna(df['cons_price_idx'].median())

print("Nulos restantes en euribor3m:", df['euribor3m'].isna().sum())
print("Nulos restantes en cons_price_idx:", df['cons_price_idx'].isna().sum())


Nulos restantes en euribor3m: 0
Nulos restantes en cons_price_idx: 0


### 2.7. Creación de nuevas variables
Añado columnas que nos servirán para el análisis posterior:
- `total_children`: suma de niños y adolescentes.
- `previously_contacted`: binaria (1 si se le contactó antes de esta campaña, 0 si no).
- `age_group`: rangos de edad (`<30`, `30-44`, `45-59`, `60+`).


In [11]:
df['total_children'] = df['kidhome'] + df['teenhome']
df['previously_contacted'] = (df['pdays'] != 999).astype(int)

df['age_group'] = pd.cut(
    df['age'],
    bins=[0, 29, 44, 59, 120],
    labels=['<30', '30-44', '45-59', '60+']
)

df['age_group'].value_counts(sort=False)


age_group
<30       5363
30-44    25456
45-59    11084
60+       1097
Name: count, dtype: int64

### 2.8. Guardar el dataset limpio
Comprobamos que todo esté correcto y guardamos en `data/processed/bank_cleaned.csv`.


In [12]:
print("Nulos pendientes (solo columnas con nulos):")
print(df.isna().sum()[df.isna().sum() > 0])
print(f"\nDimensiones finales: {df.shape}")

output_path = '../data/processed/bank_cleaned.csv'
os.makedirs('../data/processed', exist_ok=True)
df.to_csv(output_path, index=False)

print(f"Dataset guardado en {output_path}")


Nulos pendientes (solo columnas con nulos):
date             248
contact_month    248
contact_year     248
dtype: int64

Dimensiones finales: (43000, 33)
Dataset guardado en ../data/processed/bank_cleaned.csv
